## Activity 1: Test normality of data attributes (columns) and carry out

In [1]:
%matplotlib inline
import numpy as np
import pandas as pd
import seaborn as sns
sns.set_theme(style="ticks")

In [2]:
from scipy import stats
from sklearn import preprocessing

In [3]:
df = pd.read_csv('Dataset/bank.csv', sep=';')

In [4]:
DV = 'y'
df[DV]= df[DV].astype('category')
df[DV] = df[DV].cat.codes

In [5]:
msk = np.random.rand(len(df)) < 0.8
train = df[msk]
test = df[~msk]

In [6]:
# selecting the target variable (dependent variable) as y
y_train = train[DV]

In [7]:
train = train.drop(columns=[DV])
train.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome
0,30,unemployed,married,primary,no,1787,no,no,cellular,19,oct,79,1,-1,0,unknown
1,33,services,married,secondary,no,4789,yes,yes,cellular,11,may,220,1,339,4,failure
3,30,management,married,tertiary,no,1476,yes,yes,unknown,3,jun,199,4,-1,0,unknown
4,59,blue-collar,married,secondary,no,0,yes,no,unknown,5,may,226,1,-1,0,unknown
5,35,management,single,tertiary,no,747,no,no,cellular,23,feb,141,2,176,3,failure


In [8]:
numeric_df = train._get_numeric_data()

In [9]:
numeric_df_array = np.array(numeric_df)
loop_c = -1
col_for_normalization = list()

for column in numeric_df_array.T:
    loop_c+=1
    x = column
    k2, p = stats.normaltest(x) 
    alpha = 0.001
    print("p = {:g}".format(p))
        
    # rules for printing the normality output
    if p < alpha:
        test_result = "non_normal_distr"
        col_for_normalization.append((loop_c)) # applicable if yeo-johnson is used
        
        #if min(x) > 0: # applicable if box-cox is used
            #col_for_normalization.append((loop_c)) # applicable if box-cox is used
        print("The null hypothesis can be rejected: non-normal distribution")
        
    else:
        test_result = "normal_distr"
        print("The null hypothesis cannot be rejected: normal distribution")

p = 2.83447e-56
The null hypothesis can be rejected: non-normal distribution
p = 0
The null hypothesis can be rejected: non-normal distribution
p = 1.97156e-209
The null hypothesis can be rejected: non-normal distribution
p = 0
The null hypothesis can be rejected: non-normal distribution
p = 0
The null hypothesis can be rejected: non-normal distribution
p = 0
The null hypothesis can be rejected: non-normal distribution
p = 0
The null hypothesis can be rejected: non-normal distribution


In [10]:
pt = preprocessing.PowerTransformer(method='yeo-johnson', standardize=True, copy=True)

In [11]:
columns_to_normalize = numeric_df[numeric_df.columns[col_for_normalization]]
names_col = list(columns_to_normalize)

In [12]:
columns_to_normalize.plot.kde(bw_method=3)

<Axes: ylabel='Density'>

In [13]:
normalized_columns = pt.fit_transform(columns_to_normalize)
normalized_columns = pd.DataFrame(normalized_columns, columns=names_col)

In [14]:
normalized_columns.plot.kde(bw_method=3)

<Axes: ylabel='Density'>

In [15]:
numeric_df_array = np.array(normalized_columns) 
loop_c = -1

for column in numeric_df_array.T:
    loop_c+=1
    x = column
    k2, p = stats.normaltest(x) 
    alpha = 0.001
    print("p = {:g}".format(p))
        
    # rules for printing the normality output
    if p < alpha:
        test_result = "non_normal_distr"
        print("The null hypothesis can be rejected: non-normal distribution")
        
    else:
        test_result = "normal_distr"
        print("The null hypothesis cannot be rejected: normal distribution")


p = 1.13252e-18
The null hypothesis can be rejected: non-normal distribution
p = 0
The null hypothesis can be rejected: non-normal distribution
p = 2.0825e-151
The null hypothesis can be rejected: non-normal distribution
p = 0.00897784
The null hypothesis cannot be rejected: normal distribution
p = 0
The null hypothesis can be rejected: non-normal distribution
p = 3.67698e-199
The null hypothesis can be rejected: non-normal distribution
p = 3.51405e-199
The null hypothesis can be rejected: non-normal distribution


In [16]:
columns_to_notnormalize = numeric_df
columns_to_notnormalize.drop(columns_to_notnormalize.columns[col_for_normalization], axis=1, inplace=True)

In [17]:
numeric_df_normalized = pd.concat([columns_to_notnormalize.reset_index(drop=True), normalized_columns], axis=1)

In [18]:
numeric_df_normalized

,age,balance,day,duration,campaign,pdays,previous
0,-1.139033,0.269971,0.420783,-0.879160,-1.118756,-0.469972,-0.470002
1,-0.745326,1.189387,-0.541507,0.168408,-1.118756,2.149771,2.181784
2,-1.139033,0.163612,-1.731089,0.058449,1.065465,-0.469972,-0.470002
3,1.513908,-0.451799,-1.395600,0.198182,-1.118756,-0.469972,-0.470002
4,-0.505525,-0.103127,0.858380,-0.306775,0.111267,2.132796,2.176931
...,...,...,...,...,...,...,...
3635,-0.745326,-1.344659,1.576999,0.626458,1.298115,-0.469972,-0.470002
3636,1.386524,-13.476951,-0.808492,-0.221908,-1.118756,-0.469972,-0.470002
3637,1.386524,-0.291406,0.420783,-0.235651,1.864059,-0.469972,-0.470002
3638,-1.428088,0.043255,-1.240361,-0.398003,1.065465,2.138181,2.176931
